<div style="font-family:verdana;"><span style="font-size:250%;"> <center>Pipelines for Preprocessing: A tutorial</center> </span>
    
</div>

*Abstract*

This Kaggle tutorial provides a comprehensive overview of pipelines for data preprocessing in machine learning. The tutorial explains the concept of pipelines and demonstrates their practical implementation using scikit-learn. It covers essential preprocessing techniques like handling missing data, feature scaling, and one-hot encoding. The benefits of using pipelines, such as improved code organization, readability, and reusability are discussed. Additionally, the tutorial explores the integration of pipelines with cross-validation and hyperparameter tuning.  The example pipeline built here uses the new `set_output` API for scikit-learn transforms, which produces pandas output with informative column names. This tutorial will be valuable to anyone looking to enhance their preprocessing workflows with pipelines.

# <p style="background-color:darkred;color:white;font-family:verdana;font-size:120%;text-align:center;border-radius: 15px 50px;">Contents</p>

**<a href=#1.-Introduction>1. Introduction</a>**

**<a href=#2.-Scaling-and-transforming-numerical-data>2. Scaling and transforming numerical data</a>**

**<a href=#3.-Imputation>3. Imputation</a>**

**<a href=#4.-Encoding-categorical-data>4. Encoding categorical data</a>**

**<a href=#5.-ColumnTransformer---applying-different-transformations-to-different-columns>5. ColumnTransformer - applying different transformations to different columns</a>**

**<a href=#6.-Replacing-a-column-with-a-custom-function>6. Replacing a column with a custom function</a>**

**<a href=#7.-Feature-engineering---adding-a-column>7. Feature engineering - adding a column</a>**

**<a href=#8.-Drop-columns>8. Drop columns</a>**

**<a href=#9.-Named-transformer-output>9. Named transformer output</a>**

**<a href=#10.-Modelling-using-the-pipeline>10. Modelling using the pipeline</a>**

**<a href=#11.-Testing-preprocessing-choices-using-GridSearchCV>11. Testing preprocessing choices using GridSearchCV</a>**

**<a href=#12.-Limitations-of-pipelines>12. Limitations of pipelines</a>**

**<a href=#13.-Conclusions>13. Conclusions</a>**

In [ ]:
from IPython.display import Image
Image("../input/pipeline-diagrams/pipeline.png",width=1200)

*Image courtesy of DALL-E*

   
# <p style="background-color:darkred;color:white;font-family:verdana;font-size:120%;text-align:center;border-radius: 15px 50px;">1. Introduction</p>  

Pipelines are a powerful tool for organizing your machine-learning workflow. In this notebook I show how you can use scikit-learn pipelines to set up your data preprocessing in a clean and efficient way, eliminating duplicated code, minimising the chance of errors, and allowing tuning of preprocessing steps to create better predictions. 

Some common preprocessing steps that we might want to include in a pipeline are:

1. Scaling and transforming numerical data

2. Encoding a categorical column

3. Imputing missing values

4. Processing and replacing a column

5. Adding columns derived from one or more columns (e.g. indicator and aggregation columns)

6. Dropping columns

By setting these processing steps as scikit-learn transformers (rather than processing the data with plain functions), we can apply these data transformations more efficiently. 

## 1.0 Incorrect/inefficient approaches

Too often here on kaggle we see preprocessing following one of two workflows, firstly appending the test data to the training data and applying preprocessing to the combined dataset. A simple example, is encoding a categorical variable in both training and test data:

> ```
> both = pd.concat([train.drop(target_variable, axis=1), test])
> both[categorical_variable] = OrdinalEncoder.fit_transform(both[categorical_variable])
> train = both.iloc[train.shape[0]:,:]
> test = both.iloc[:train.shape[0],:]
> ```

The main reason this is a bad idea is because of data leakage. Basically we are building a substandard model by applying preprocessing to training and test data simultaneously as information from the test data can leak out into the preprocessed training data. (See section 1.2.)

Another common approach is simply applying the preprocessing twice:

> ```
> oe = OrdinalEncoder()
> train[categorical_variable] = oe.fit_transform(train[categorical_variable])
> test[categorical_variable] = oe.transform(test[categorical_variable])
> ```

Done properly (i.e. calling `transform` rather than `fit_transform` on the test data), this eliminates the data leakage problem. 

However, if we're using a cross-validation scheme, this should be applied the training and test data *within each CV fold*. Thus, this approach can suffer from issues with repeated code, as we'd need to apply the transform multiple times within the workbook. Moreover, if there is more than one pre-processing step, combining all the steps into a pipeline means that we only need to perform the `fit` and `transform` (or `predict`) methods on the pipeline rather than each individual step. 

## 1.1 What is a pipeline?

A pipeline is a series of processing and/or modelling steps bundled together into one object. The [Kaggle Learn "Pipelines" tutorial](https://www.kaggle.com/code/alexisbcook/pipelines) provides a short introduction to pipelines.

The [scikit-learn documentation](https://scikit-learn.org/stable/modules/compose.html#pipeline) states that:

> Pipeline can be used to chain multiple estimators into one.

Here, an 'estimator' typically means a data transformer (e.g. `StandardScaler`, or `OneHotEncoder`), or a predictive model (such as `LinearRegression`). The most common use case is chaining a number of data preprocessing steps together before a model. Typically, we have a number of transformers that are fitted and then applied to the data, and a predictive model at the end of the pipeline chain. Again from the [scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html):

> Intermediate steps of the pipeline must be ‘transforms’, that is, they must implement `fit` and `transform` methods. The final estimator only needs to implement `fit`.

Often the final estimator has `predict` and `score`, etc. methods as well. Alternatively we could have a chain of transformations without a model at the end, and we'd typically call `transform` or `fit_transform` on the pipeline. 

From the [scikit-learn pipeline documentation](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html):

> The purpose of the pipeline is to assemble several steps that can be cross-validated together while setting different parameters. 

Quite often we have a series of transformations (e.g. encoding categorical variables, power transforms for numerical variables, feature engineering, imputing missing values) that we'd like to apply to a dataset, and a candidate model (or models). Notice that preprocessing steps in scikit-learn all use `fit()` and `transform()` methods, and the machine learning estimators and models have similarly structured APIs, namely the `fit()` and `predict()` methods. The idea behind a pipeline is to use these similar APIs to link together transformations and models so that the output of one step feeds into the input of the next step in the pipeline.

Note that when `fit` is called on the pipeline, each step is fitted and transformed, whereas when `predict` (or `score`, `transform`, etc.) is called on the entire pipeline, each estimator is only transformed, not fitted. 

In [ ]:
from IPython.display import Image
Image("../input/pipeline-diagrams/pipeline_schematic.png",width=800,height=500)


## <div id="why">1.2 Why pipelines?</div>

There are several benefits of using a pipeline, and these include:

1. Eliminating duplicated code.

    An important tenet of software engineering is ["Don't repeat yourself"](https://en.wikipedia.org/wiki/Don%27t_repeat_yourself), or DRY. Copying and pasting code used to process training data to process the testing data is fraught with problems. Changes made to the data pre-processing workflow can be easily missed or errors made if you need to change the code in two (or more) places. Using a pipeline eliminates duplicated code by combining preprocessing steps into one class which can be easily instantiated again.

2. Avoiding [data](https://machinelearningmastery.com/data-preparation-without-data-leakage/) [leakage](https://jfrog.com/community/data-science/be-careful-from-data-leakage/) by processing training and test data separately.

    One (substandard) way to get around duplicated code is to combine the training and test data and apply the preprocessing to the combined dataset. I used to do this, and I see it quite commonly on kaggle, but it is [not recommended](https://community.alteryx.com/t5/Data-Science/Dealing-with-Data-Leakage/ba-p/827583). Briefly, preprocessing training and test data concurrently means test data information (e.g. distributions of features) is seen by the `fit()` method, and this will provide an overly optimistic test score, which ultimately leads to degraded performance on new predictions.
    
    In contrast, we see from the above diagram that when the pipeline's `predict` (or `score`, `transform`, etc.) method is called, the pipeline is not re-fitted on the test dataset, but the pipeline is just transformed without fitting. Thus we avoid data leakage from preprocessing using a pipeline in this way.

3. Embedding preprocessing in a cross-validation scheme.

    We saw in section 1.0 that data leakage from preprocessing can be avoided by calling `fit_transform` on the training data and `transform` on the test data. However, if we're using a cross-validation scheme, the preprocessing should be applied in each fold, first being fitted to the CV training data and then applied (via `transform`) to the CV test data. This can be accomplished with model validation schemes like `cross_validate` or `GridSearchCV` by passing the pipeline in as the estimator, rather than the final model, and the raw data as 'X' and 'y'. 

4. Code readability.

    By eliminating duplicated code and having everything in one place, readability of your code will be improved. 

5. Allows the possibility of tuning preprocessing choices for better test predictions.

    Do you know what the effect is of the different preprocessing choices you've made on your model predictions? Testing the effect of preprocessing is often done manually, by making a change and rerunning the notebook. This is slow and inconvenient to do, prone to errors, and it's very difficult to perform a proper workflow evaluation like this. Pipelines can automate the tuning and evaluation of your preprocessing workflow and help to answer questions like:
    
    1. Do particular preprocessing steps actually benefit the model?
    2. For categorical encoding (e.g. OneHotEncoder), does grouping smaller infrequent classes help?
    3. What parameters are best for numerical transforms (e.g. power transforms)?
    4. Does dropping certain columns help predictions?
    
    


In [ ]:
import numpy as np 
import pandas as pd 

import matplotlib.pyplot as plt
import seaborn as sns

sample = pd.read_csv('/kaggle/input/titanic/gender_submission.csv')
train = pd.read_csv('/kaggle/input/titanic/train.csv', index_col = 'PassengerId')
trainX = train.drop(['Survived'], axis=1)
trainy = train['Survived']
test = pd.read_csv('/kaggle/input/titanic/test.csv', index_col = 'PassengerId')

## 1.3 Example data - survial on the Titanic

To demonstrate the use of pipelines to perform data preprocessing, we'll use the titanic dataset. Here we need to predict whether a passenger survived based on the information provided (Age, Sex, Name, etc.).

In [ ]:
Image("../input/pipeline-diagrams/titanic.png",width=800,height=500)

*Image courtesy of DALL-E*

Let's have a closer look at the data:

In [ ]:
train.head()

The variables 'Sex', 'Cabin' and 'Embarked' are all string (object) variables, and we'll definitely need to do something with them (i.e. encoding) before a numerical model can be used. There is potentially some information in 'Ticket', 'Name' and 'Cabin' that could be recovered through feature engineering, such as the title ('Mr.', 'Miss.', etc.). 

What about missing values?

In [ ]:
print('Number of missing values per column')
pd.concat([trainX.isna().sum(0),test.isna().sum(0)], axis=1).rename({0: 'train', 1:'test'}, axis=1)

The analysis shows that 'Age' and 'Cabin' have missing values in both train and test datasets, whereas 'Embarked' and 'Fare' have missing values in the train and test datasets respectively. These features will need to be imputed.

Additionally, we may wish to:
- drop some columns
- create new columns (i.e. feature engineering)
- scale the data (e.g. normalization, standardization or a power transform)

Finally, other data transformations may also be useful (e.g. principal component analysis, or PCA).

With a pipeline we can wrap all of this (and much more!) in the one estimator. 


# <p style="background-color:darkred;color:white;font-family:verdana;font-size:120%;text-align:center;border-radius: 15px 50px;">2. Scaling and transforming numerical data</p>

Let's begin with the simplest pipeline: a data processing step and a model (actually, the simplest pipeline we can construct has just one step, but that's not much use). Let's build a model on the titanic dataset to predict 'Surived' based on some of the numerical features: 'Pclass', 'SibSp', 'Parch', 'Fare'. These features have no missing values and require no encoding. Thus, we can apply a power transform directly to these variables directly, and then model the transformed data directly.


In [ ]:
plt.figure(figsize=(16,4))
plt.subplots_adjust(wspace=0.4)
for i,x in enumerate(['Pclass','SibSp', 'Parch', 'Fare']):
    plt.subplot(1,4,i+1)
    sns.histplot(train[x])
    group_mean = train.groupby(train[x])['Survived'].mean()
    ax = plt.gca()
    ax2=ax.twinx()
    sns.regplot(x=group_mean.index, y=list(group_mean),lowess=True,scatter=False, color='red')
    if i == 0:
        ax.set_ylabel('Count')
    else:
        ax.set_ylabel('')
    if i == 3:
        ax2.set_ylabel('\nAverage survival \nprobability (smoothed)')
    

It is evident from histograms of the features we selected that these four variables are skewed. Normalization of these variables could help improve predictions. Sklearn has a transformer (`PowerTransformer`) that can perform this for us. After transforming 'Fare', the distribution is more even, and it looks like the error in the logistic regression (just for this variable) is reduced.

In [ ]:
from sklearn.preprocessing import PowerTransformer

f, axes = plt.subplots(2, 2, gridspec_kw={'height_ratios': [3,1]},
                      figsize=(12,6))
_=sns.regplot(x=train['Fare'], y=trainy, logistic=True, line_kws={'color': 'red'},
           ax=axes[0,0])
_=sns.regplot(x=PowerTransformer().fit_transform(train[['Fare']]), y=trainy, logistic=True, 
           line_kws={'color': 'red'},
        ax=axes[0,1])
_=plt.gca().set_xlabel('Transformed Fare')
_=sns.histplot(x=train['Fare'],
            ax = axes[1,0])
_=plt.gca().set_xlabel('Fare')
_=sns.histplot(x=PowerTransformer().fit_transform(train[['Fare']]).ravel(),
            ax = axes[1,1])
_=plt.gca().set_xlabel('Transformed Fare')
             

Now, we can test the effect of applying the power transformation on the four variables before fitting a logistic regression by first fitting a `LogisticRegression` on the raw data. Call `cross_validate` to get an estimate of the cross-validated error from using logistic regression. We use `estimator=LogisticRegression()`:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, StratifiedKFold, KFold

cv_mean_acc = cross_validate(estimator=LogisticRegression(), 
               scoring = 'accuracy',
               cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123), 
               X = train[['Pclass','SibSp', 'Parch', 'Fare']], y=trainy)['test_score'].mean()
print(f"Average cross-validated accuracy from logistic regression on raw data: {cv_mean_acc:.3f}")

To define a pipeline, we just put the steps into a list and call the `Pipeline` constructor:

In [ ]:
from sklearn.pipeline import Pipeline

simplest_pipeline = Pipeline(steps = [('normalize', PowerTransformer()),
                                      ('lr_model', LogisticRegression())])

Each step has a name (here 'normalize' and 'lr_model') and the associated transformer/estimator. The names can be anything, but we'll see later that we use these names to set and retrieve parameters for each step of the pipeline using the `set_params()` and `get_params()` methods, and also through cross-validation schemes like `GridSearchCV`.

To fit the logistic regression on the transformed data, we pass the pipeline as the estimator in `cross_validate` along with the raw data, rather than the model instance (i.e. `LogisticRegression`) and the transformed data:

In [ ]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=124)
cv_mean_acc_pipeline = cross_validate(estimator=simplest_pipeline,
               scoring = 'accuracy',
               cv = skf,
               X = train[['Pclass','SibSp', 'Parch', 'Fare']], y=trainy)['test_score'].mean()
print(f"Average cross-validated accuracy from logistic regression with transformed data: {cv_mean_acc_pipeline:.3f}")

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import FeatureUnion

Success! The cross-validated accuracy is improved when we include a power transformation before the logistic regression.

We could apply other numerical transformations (such as principal component analysis: `sklearn.decomposition.PCA`, and scaling, e.g. `sklearn.preprocessing.StandardScaler`) in a similar way.

<div class="alert alert-block alert-warning">
<b>NB:</b> The pipeline workflow allows us to fit and transform the training and test data <i>within</i> the cross-validation scheme used by cross_validate, whereas cross validating the logistic regression model directly and passing the pre-transformed data as parameters 'X' and 'y' in the call to cross_validate can result in data leakage.</div>



In [ ]:
class SimpleImputerNamed(SimpleImputer):
    from sklearn.impute import SimpleImputer
    def get_feature_names_out(self):
        return list(self.feature_names_in_)
class OrdinalEncoderNamed(OrdinalEncoder):
    def get_feature_names_out(self):
        return list(self.feature_names_in_)
class OneHotEncoderNamed(OneHotEncoder):
    def get_feature_names_out(self):
        names_out = []
        for i, name_in in enumerate(self.feature_names_in_):
            names_out += [f'{name_in}_{j}' for j in self.categories_[i]]
        return names_out
class ColumnTransformerNamed(ColumnTransformer):
    def get_feature_names_out(self):
        names = []
        for transformer in self.transformers_:
            if transformer[0] == 'remainder':
                if transformer[1] == 'passthrough':
                    names += list(self.feature_names_in_[transformer[2]])
                break
            else:
                names += transformer[1].get_feature_names_out()
        return names
    def fit(self, X, y=None):
        #print('In fit method')
        return super().fit(X,y)
    def transform(self, X):
        #print('In transform method')
        transformed = super().transform(X)
        return pd.DataFrame(transformed, columns= self.get_feature_names_out())
    def fit_transform(self, X, y=None):
        #print('In fit_transform method')
        fit_transformed = super().fit_transform(X,y)
        return pd.DataFrame(fit_transformed, columns=self.get_feature_names_out())
    


class Identity(TransformerMixin, BaseEstimator):
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        return np.array(X)
class IdentityNamed(Identity):
    def fit(self, X, y=None):
        self.feature_names_in_ = list(X.columns)
        return self
    def get_feature_names_out(self):
        return self.feature_names_in_
    
class FeatureUnionNamed(FeatureUnion):
    def __init__(self, transformer_list):
        self.transformers_ = transformer_list
        super().__init__(transformer_list)
    def get_feature_names_out(self):
        names = []
        for transformer in self.transformers_:
            names += transformer[1].get_feature_names_out()
        return names
    def fit(self, X, y=None):
        return super().fit(X,y)
    def transform(self, X):
        transformed = super().transform(X)
        return pd.DataFrame(transformed, columns= self.get_feature_names_out())
    def fit_transform(self, X, y=None):
        print(X.shape)
        fit_transformed = super().fit_transform(X,y)
        return pd.DataFrame(fit_transformed, columns=self.get_feature_names_out())
    



# <p style="background-color:darkred;color:white;font-family:verdana;font-size:120%;text-align:center;border-radius: 15px 50px;">3. Imputation</p>

In a general sense, imputation can be thought of as the "filling in" of missing values by some scheme. The [Kaggle Learn "Handling Missing Values" Tutorial](https://www.kaggle.com/code/alexisbcook/handling-missing-values) has a very good introductory notebook on imputation. 

Imputation strategies can range from simple to very complex. For this example, we will impute in a simple way by filling in with the mean value of non-missing data. 

Take the 'Age' column as an example. There are 177 and 86 missing rows in the training and test datasets, and the mean from the non-missing training data is 29.7 years:

In [ ]:
print(f'Mean of non-missing Age column in training dataset: {train["Age"].mean():.1f}')
print(f'Mean of non-missing Age column in test dataset: {test["Age"].mean():.1f}')

## 3.1 Direct imputation

A direct (naive) way to do this is to fill in the values using the index values from a call to `isna()`:

>```
> train_copy = train.copy()
> train_copy.loc[train_copy['Age'].isna(), 'Age'] = train_copy['Age'].mean()
>```

To apply this to the test dataset, we would need to write something like:

>```
> test[test['Age'].isna(), 'Age'] = train['Age'].mean()`
>```

However, this violates the DRY coding principle in that we are repeating the same statement but with different datasets. Errors can quickly creep in when with this, either when the original code is changed (e.g. we decide to infill with a median), or we forget to change `train` to `test` when modifying the copied code. We can do better than this! 

<div class="alert alert-block alert-danger">
    <b>Warning!</b> Don't impute like this! A direct approach like this suffers from all of the problems identified in section 1.2 Why pipelines?</div>

## 3.2 Transformer-based method of imputation

The direct (procedural) way of imputing as described above can't really be implemented in a pipeline. Since pipelines are built from scikit-learn transformers, we need a transformer-based way of filling in the missing values. Luckily, scikit-learn has a transformer that does exactly this, namely `SimpleImputer`. We use this by calling `fit` on the training data, then `transform` on the training data and then `transform` on the test data. Alternatively, the first two steps can be combined with a call to `fit_transform`:

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')
imputed_train_Age = imputer.fit_transform(train[['Age']])
imputed_test_Age = imputer.transform(test[['Age']])

The output `imputed_train_Age` and `imputed_test_Age` are exactly what we want, i.e. arrays (or columns) of 'Age' with missing values filled in. Notice that we use the same object (`imputer`) to apply the imputation to both training and test datasets, so we are not violating the DRY principle.

In [ ]:
print(f'Mean of imputed_train_Age: {imputed_train_Age.mean():.1f}')
print(f'Mean of imputed_test_Age: {imputed_test_Age.mean():.1f}')

<div class="alert alert-block alert-warning">
<b>NB:</b> Notice that whereas the mean of imputed 'Age' from the training data is equal to the mean of non-missing 'Age' from the training data, the mean of imputed 'Age' from the test data is different to the mean of non-missing 'Age' from the test data. This is because we have infilled the test data using the mean from the training data, by only calling transform (not fit) on the test data.</div>

## 3.3 Imputation pipeline

Now that imputation is set up with a transformer, we can include this (either `imputer` or `SimpleImputer(strategy='mean')` directly) as a pipeline step, e.g.

>```
> imputation_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='mean')),
>                                       ('model', LogisticRegression())])
> imputation_pipeline.fit(train[['Age']], trainy)    
>``` 

Notice that we only pass through `train[['Age']]` at this point as each step of the pipeline is applied to the entire data set, unless we use a `ColumnTransformer` (see below).

# <p style="background-color:darkred;color:white;font-family:verdana;font-size:120%;text-align:center;border-radius: 15px 50px;">4. Encoding categorical data</p>

The two categorical columns 'Sex' and 'Embarked' will need to be encoded for use in most models. The [Kaggle Learn "Categorical Variables" Tutorial](https://www.kaggle.com/code/alexisbcook/handling-missing-values) provides an excellent introduction to encoding with ordinal and one-hot encoding strategies. Here, we'll use one-hot encoding to encode both columns. 

One way of doing this is with pandas [get_dummies()](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html) function (in theory we could do it manually as well...), but since we are trying to set up a pipeline it is best to consider a scikit-learn transformer approach to doing this.

## 4.1 Encoding with the OneHotEncoder transform

Applying the `OneHotEncoder` transform directly is fairly straight-forward. There are a couple of options available, which are described in the [user manual](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html). I usually setting the parameter `handle_unknown` to 'ignore', because it there's a category in the test data that isn't in the training data, the pipeline won't throw an error. 

In [ ]:
from sklearn.preprocessing import OneHotEncoder
one = OneHotEncoder(sparse_output=False,
                    handle_unknown='ignore')
pd.concat([train['Sex'], pd.DataFrame(one.fit_transform(train[['Sex']]),
                                      index=train.index)], axis=1)


Here, the 'Sex' column has been split into two indicator variables, one each for male and female.

Similarly to the imputation section above, we could set up a pipeline that operates on just the categorical columns:

> ```
> encoding_pipeline = Pipeline(steps=[('encode', OneHotEncoder(handle_unknown='ignore')),
>                                     ('model', LogisticRegression())])
> encoding_pipeline.fit(train[['Sex', 'Embarked']], trainy)    
>```

In reality, though, we probably want to impute some columns, encode other columns, and maybe do nothing to other columns. The example pipelines we have set up so far only work on specific columns. In order to selectively apply transformers to different columns we need `ColumnTransformer`:

# <p style="background-color:darkred;color:white;font-family:verdana;font-size:120%;text-align:center;border-radius: 15px 50px;">5. ColumnTransformer - applying different transformations to different columns</p>

In any practical application, there are going to be different types of columns, which need different treatments. `ColumnTransformer` selectively applies transformers to different columns. We saw that the basic syntax of pipelines in scikit-learn was a list of steps. For each step we specify its name (which can be anything we like) and the transformer itself in a tuple. [The syntax](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) for `ColumnTransformer` is similar but we also need to specify the relevant columns for each step as well in the third position of the tuple. 

In [ ]:
from sklearn.compose import ColumnTransformer

multicolumn_prep = ColumnTransformer([('impute', 
                                       SimpleImputer(strategy='mean'), 
                                       ['Age', 'Fare']),
                                      ('encode', 
                                       OneHotEncoder(handle_unknown='ignore'), 
                                       ['Sex', 'Embarked']),
                                     ],
                                     remainder='passthrough')
multicolumn_prep

Two things to note about the `ColumnTransformer` defined here are that: 1) we don't need to impute 'Embarked' as the encoding will encode missing values ('nan') as a separate category, and 2) any column not specified in the transformer list (such as 'Pclass') will 'passthrough' the `ColumnTransformer` as specified by the `remainder` argument, i.e. these variables will stay untransformed.

Now, we can include this as the preprocessing step in a pipeline.

In [ ]:
ct_pipeline = Pipeline([('preprocessing', multicolumn_prep),
                        ('lr_model', LogisticRegression(max_iter = 2000))])
ct_pipeline

As our pipeline gets more and more complicated, it is worth pointing out at this point that we can get an interactive flowchart representation of the pipeline through the `set_config` function:

In [ ]:
from sklearn import set_config
set_config(display="diagram")
ct_pipeline


Now, let's get an estimate of the accuracy from this preprocessing step combined with the logistic regression:

In [ ]:
ct_cv_res = cross_validate(estimator = ct_pipeline, 
                           X = trainX.drop(['Name', 'Ticket', 'Cabin'],
                                           axis=1),
                           y = trainy,
                           cv = skf,
                           scoring = 'accuracy')['test_score'].mean()
print(f"Average cross-validated accuracy from\ncolumn transformer pipeline: {ct_cv_res:.3f}")

Pretty good! These four variables encoded properly provide a pretty good model, better than the transformed numerical model in section 2.

## 5.1 Encoding based on variable dtype

Rather than specifying the column names directly as above, we can also use `make_column_selector` to automatically select columns based on their 'dtypes', in this case it makes sense to encode everything that's an 'object' dtype, and leave other columns (i.e. numerical dtypes). 

In [ ]:
from sklearn.compose import make_column_selector

encode_categoricals = ColumnTransformer([('encode_cats',
                                          OneHotEncoder(handle_unknown='ignore'), 
                                          make_column_selector(dtype_include='object')),
                                        ],
                                        remainder='passthrough')
encode_categoricals.fit_transform(train[['Age', 'Fare', 'Sex', 'Embarked']])

<div class="alert alert-block alert-warning">
<b>NB:</b> When using 'make_column_selector', make sure you know what columns are being transformed. Encoding inappropriate variables with many levels (e.g. date columns) can result in big performance issues.</div>

# <p style="background-color:darkred;color:white;font-family:verdana;font-size:120%;text-align:center;border-radius: 15px 50px;">6. Replacing a column with a custom function</p>

The preprocessing transformers available with `sklearn.preprocessing` can do lots of things, but not everything. Processing of string (object) feature will in general require a custom function to extract and transform the relevant information. As an example, let's process the 'Cabin' feature.

In [ ]:
train['Cabin']

This column contains either the cabin number, e.g. 'C85', or NaN if the passenger wasn't in a cabin. The cabin number looks to consist of a deck 'A', 'B', 'C' and so on, and a number. Under the assumption that the deck could contain predictive information but the number on the deck won't, let's consider extracting the initial letter from this feature. 

## 6.1 List-comprehension approach

Extracting the deck letter is slightly hampered by the fact that NaN is considered a number (i.e. a `float`) but the cabin ID 'C85' is a string. We can use a conditional list comprehension to get what we want:

In [ ]:
[x[0] if type(x) == str else 'None' for x in train['Cabin']][:15]

We can process this into a kind of ordinal encoding, so that it can be used directly, as follows:

In [ ]:
[ord(x[0]) - ord('A') + 1 if type(x) == str else 0 for x in train['Cabin']][:15]

Here, 0 stands for no cabin, 1 for 'A', etc. 

To incorporate this text extraction step into a pipeline we need to use a transformer rather than a function (or list comprehension). Without a pipeline we could use something like:

> ```
> train['Cabin'] = [x[0] if type(x) == str else 'None' for x in train['Cabin']]
> ```

but this approach is discouraged as discussed in the introduction section.

## 6.2 FunctionTransformer approach

In the same way that we replaced a direct approach to imputation with a scikit-learn transformation approach in section 3.2, we replace the list-comprehension method of extracting the deck letter from 'Cabin' with a `FunctionTransformer`, which allows us to define a transformer based on an arbitrary function. 

In [ ]:
extract_cabin = FunctionTransformer(func = lambda col: np.array([ord(x[0]) - ord('A') + 1 \
                                                                 if type(x) == str else \
                                                                 0 for x in col])[:,np.newaxis])
extract_cabin.fit_transform(train['Cabin'])[:15]


You may need to playing around a bit with the functional form of the transformation so that it outputs an array (column) rather than a 1-D list. To use the `extract_cabin` transformer in a `ColumnTransformer` we need an array so that it can be column-bound with the rest of the output from the transformers. The list comprehension that we developed operates on a single column (`col`), and returns a length-n list. 

The [example from the FunctionTransformer documentation](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.FunctionTransformer.html) uses a numpy array function (`np.log1p`), which returns an array from an array. In the `FunctionTransformer`, we cast the list as an array and add an extra dimension (using `np.newaxis`) to get an (n, 1) shaped array, which will work with the rest of the steps from `ColumnTransformer`.

Now, we can combine this with the imputation and encoding in `ColumnTransformer`:

In [ ]:
multicolumn_prep_with_cabin = ColumnTransformer([('impute', 
                                                  SimpleImputer(strategy='mean'), 
                                                  ['Age', 'Fare']),
                                                 ('cabin_extract', 
                                                  extract_cabin, 
                                                  'Cabin'),
                                                 ('encode', 
                                                  OneHotEncoder(handle_unknown='ignore',
                                                               sparse_output=False), 
                                                  ['Sex', 'Embarked']),
                                                ],
                                                remainder='passthrough')
multicolumn_prep_with_cabin

Notice that the specified column for the `extract_cabin` transformer is `'Cabin'` rather than `['Cabin']`, and this is due to the way the `FunctionTransformer` is set up, if we had a true array function (like the [example in the sklearn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.FunctionTransformer.html)), we could have the columns to be transformed in a list as with the other steps in the `ColumnTransformer`. See also [this discussion on stack overflow](https://stackoverflow.com/questions/56298242/valueerror-input-array-dimensions-not-right-for-countvectorizer/56299794#56299794) about this problem.

<div class="alert alert-block alert-info"><b>Tip:</b> it is best to use an array function in FunctionTransformer</div>

Now, let's have a look to see if this improves our modelling:

In [ ]:
cv_with_cabin = cross_validate(estimator = Pipeline([('preprocessing', multicolumn_prep_with_cabin),
                                                     ('lr', LogisticRegression(max_iter=2000))]),
                               X = trainX.drop(['Name', 'Ticket'], 
                                           axis=1),
                               y = trainy,
                               cv = skf,
                               scoring = 'accuracy')['test_score'].mean()
print(f"Average cross-validated accuracy including 'Cabin' feature engineering: {cv_with_cabin:.3f}")

## 6.3 Custom transformer approach

More flexibility can be obtained by using custom transformers, although it is often a little bit more work to set up the class. Most of the work we need to do is in writing the `fit()` and `transform()` methods, whch is where most of the action happens. 

We can get most of what we need to make a pipeline-ready transformer by inheriting from `TransformerMixin` and `BaseEstimator`. This includes `__init__()`, `fit_transform()` (which just runs `fit()` and then `transform()`), and `get_params()` and `set_params()` (which we need if we're using the pipeline with `GridSearchCV`, etc).

If we'd like to specify a parameter for the transformation, we need to override the default `__init__()` method. [Setting parameters](https://scikit-learn.org/stable/developers/develop.html#parameters-and-init) in the `__init__()` method means we can set these parameters in the call to the class (like the way we specified mean imputation using `SimpleImputer(strategy='mean')` above).

In a general sense, the `fit()` method looks at the input ('X') and fits parameters, such as minimum and maximum for a scaling transformer, or categories for an encoding transformer. Note that `fit()` methods of scikit-learn transformers need to end with `return self` to work well with the rest of the scikit-learn architecture.

The `transform()` method is where we actually apply the transformation to the input data (both train and test).

### 6.3.1 Extract title from 'Name'

Although the names of passengers might not be directly useful, one idea is to extract the title (Mr, Miss, etc.) from the name, which we can encode, and perhaps this may be useful to a model. 

In [ ]:
Image("../input/pipeline-diagrams/titles.png",width=800)

*Image courtesy of DALL-E*

We pass in a 'min_relative_frequency' parameter into the call to the class (through `__init__()`), which thresholds the titles, replacing uncommon titles (such as 'Jonkheer') with 'Rare/Unknown'

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
class ExtractTitle(BaseEstimator, TransformerMixin):
    def __init__(self, min_relative_freq):
        self.min_relative_freq = min_relative_freq
    def fit(self, X, y=None):
        from collections import Counter
        title_freq = Counter()
        titles = [x.split(',')[1].split('.')[0].strip() for x in X['Name']]
        title_freq.update(titles)
        self.common_titles = [x[0] for x in title_freq.items() if x[1] > X.shape[0]*self.min_relative_freq]
        return self
    def transform(self, X, y=None):
        title = [x.split(',')[1].split('.')[0].strip() for x in X['Name']]
        X_copy = X.copy()
        X_copy['Title'] = [x if x in self.common_titles else 'Rare/Unknown' for x in title]
        return X_copy.drop(['Name'], axis=1)

Rather than combining our transformer with `ColumnTransformer` as we did in section 6.2, to replace the 'Name' column with 'Title', we add the processed column onto the dataframe and then drop the original column directly in the `transform()` method. The custom transformer approach is more flexible then the `FunctionTransformer`/`ColumnTransformer` combination, but the transformers usually end up being more specific to the task at hand, and hence less reusable.

In [ ]:
xtract_title = ExtractTitle(min_relative_freq = 0.2)
xtract_title.fit_transform(train)

Here the 'Name' column has been replaced with the 'Title' column.

### 6.3.2 Discretizing the 'Fare' numerical column

As another example of using a custom transformer, we can bin numerical columns. Rather than having 'Fare' as a numerical (continuous) variable, we can discretize (bin) it, to convert it into an ordinal column. Since 'Fare' is highly skewed, we bin this in quantile-space, using the `pd.qcut` function

In [ ]:
class QCutFare(BaseEstimator, TransformerMixin):
    def __init__(self, n_bins):
        self.n_bins = n_bins
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        transformed = pd.qcut(X['Fare'], self.n_bins, labels=False)
        transformed[transformed.isna()] = 0
        transformed_series = pd.Series(transformed, 
                                       name=f'QCut{self.n_bins}_Fare',
                                       index=X.index)
        X_copy = X.copy()
        return pd.concat([X_copy, transformed_series], axis=1).drop(['Fare'], axis=1)
        
QCutFare(13).fit_transform(train)

The 'Fare' column is replaced with the 'QCut13_Fare' column, which specifies which quantile-bin the fare belongs to.

# <p style="background-color:darkred;color:white;font-family:verdana;font-size:120%;text-align:center;border-radius: 15px 50px;">7. Feature engineering - adding a column</p>

Adding columns can be a useful way of extracting extra information from a dataset. Creating new columns containing summaries (mean, variance, etc.) of multiple columns can provide a lot of predictive information. The best way to do this might be to calculate the statistic in a new column, but to also retain the columns (`ColumnTransformer`-based approaches that we have used so far will replace columns with the transformation). 

Custom transformers can achieve this by concatenating columns onto the 'X' matrix in the `transform` step. Let's take a closer look at the 'Age' column:

In [ ]:
np.unique(train['Age'])

We see that there are a number of fractional ages for babies below 12 months old, and, for some reason, there are a number of half ages (e.g. 20.5). It may be that people with fractional ages had more or less chance of survival, so let's engineer an indicator variable for this feature. We keep the 'Age' column itself as it is one of the more useful features.

## 7.1 Add a fractional-age indicator column

### 7.1.1 List-comprehension approach

Essentially, we can create a fractional-age indicator with the list comprehension `[int(x)!=x for x in train['Age']]`, except for the fact we have missing values. Count missing ages ('nan') as non-fractional ages:

In [ ]:
fractional_age = [0 if np.isnan(x) else 1*(int(x)!=x) for x in train['Age']]
ismissing_age = [1 if np.isnan(x) else 0 for x in train['Age']]
pd.DataFrame({'Age':train['Age'], 'FracAgeInd': fractional_age, 'AgeMissInd':ismissing_age}).iloc[105:115,:]

And we can combine these as

In [ ]:
pd.DataFrame({'Age':train['Age'],
              'FractionalAge':[0 if np.isnan(x) else 1*(int(x)!=x) for x in train['Age']]}).loc[105:115,:]

### 7.1.2 Transformer approach

Unfortunately, scikit-learn hasn't really made a useful transformer for *adding* a column to a data frame (maybe we'll see something useful in future versions?). Using `FunctionTransformer` and `ColumnTransformer` replaces columns with their transformations, so these are not useful for this.

As far as I'm aware, we need to create a custom transformer to add columns to a dataset in a pipeline. A [medium article by Xinqian Zhai](https://medium.com/mlearning-ai/select-columns-and-add-new-columns-in-an-ml-pipeline-with-code-example-bd90ccba1891) does something similar in section 3 of that article. This is similar to what we did in section 6.3, except here we don't want to drop the orignal column.

Here's a translation of the list-comprehension approach into a scikit-learn transformer class. In a similar way to the way we added 'Title' onto the dataframe in the `transform()` method of `ExtractTitle`, we append 'FractionalAge' in `transform()`:

In [ ]:
from sklearn.base import TransformerMixin, BaseEstimator
class AddFractionalAgeColumn(TransformerMixin, BaseEstimator):
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        return pd.concat([X, pd.Series([0 if np.isnan(x) else 1*(int(x)!=x) for x in X['Age']],
                                       index=X.index,
                                       name = 'FractionalAge')], axis=1)

Let's see the transformer in action:

In [ ]:
AddFractionalAgeColumn().fit_transform(train).iloc[105:115,:]

The output in the 'FractionalAge' column is identical to the output obtained from the list-comprehension approach in section 7.1.1.

## 7.2 Column aggregation - Family size

Calculating summaries (mean, variance, count, etc.) of multiple columns can be a powerful way to engineer new features. We can calculate the size of the family on board for each passenger by adding 'SibSp' and 'Parch' (plus one for the passenger itself). A transformer adding columns specified in the 'columns' parameter follows:

In [ ]:
class AddSumOfColumnsColumn(BaseEstimator, TransformerMixin):
    def __init__(self, columns, constant=0, name = 'AddedColumns'):
        self.columns = columns
        self.constant = constant
        self.name = name
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        X_copy = X.copy()
        added_series = X[self.columns].sum(1) + self.constant
        added_series.name = self.name
        return pd.concat([X, added_series], axis=1)

In [ ]:
AddSumOfColumnsColumn(columns=['SibSp','Parch'],
                      constant=1,
                      name='FamilySize').fit_transform(train)

<div class="alert alert-block alert-info"><b>Tip:</b> it is easiest to add columns in a pipeline using a custom transformer.</div>

## 7.3 Alternative method: FeatureUnion

Obviously these classes are very specific to the task at hand. Extending this approach, we might like to create more general transformer classes that add columns according to different transformations and working on different (or variable) columns. A more general way of doing this could be to use `FeatureUnion`, which creates several features from consecutive passes through a data frame, see an [example in one of my previous notebooks](https://www.kaggle.com/code/nnjjpp/author-identification-statistical-nlp#4.-Combining-feature-extraction-in-a-FeatureUnion). 

[As described on stack overflow](https://stackoverflow.com/q/55604249), `FeatureUnion` differs from `ColumnTransformer` in that each step of `FeatureUnion` processes the entire dataframe, whereas each step of `ColumnTransformer` processes a disjoint subset of columns. We could construct an `Identity` class that just returns the data-frame itself as one of the `FeatureUnion` steps, and additional transformations, perhaps wrapped up in `FunctionTransformer` classes.

# <p style="background-color:darkred;color:white;font-family:verdana;font-size:120%;text-align:center;border-radius: 15px 50px;">8. Drop columns</p>

Notice that in the pipelines we have constructed so far, we have dropped the columns we are not interested in (like 'Name' and 'Ticket' in section 6.2) before passing the data through into the pipeline. Some custom transformers drop a column in the `transform` method (such as `ExtractTitle` in section 6.3.1). 

We could alternatively incorporate column dropping into the pipeline itself.

Why would we want to do this? Well, we may want to test the effect of dropping columns on our final model using `GridSearchCV`. Most machine-learning models these days are pretty robust to the presence of uninformative columns, but predictions *can* be improved by pruning off useless columns. 

The following transformer does exactly this, and we can use it as a step in the pipeline. We put the column names to drop in the constructor.

In [ ]:
class DropColumn(BaseEstimator, TransformerMixin):
    def __init__(self, cols=[]):
        self.cols = cols
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        return X.drop(self.cols, axis=1)

<div class="alert alert-block alert-warning">
<b>NB:</b> Dropping columns that are specified in a down-stream ColumnTransformer will cause problems. If the ColumnTransformer looks for a column that has been dropped, a ValueError will occur. I discuss a work-around to this in <a href="https://stackoverflow.com/a/75170410">an answer on stack overflow</a></div>

# <p style="background-color:darkred;color:white;font-family:verdana;font-size:120%;text-align:center;border-radius: 15px 50px;">9. Named transformer output</p>

Let's combine the transformers we've developed so far into a pipeline:

In [ ]:
preprocessing_pipeline = Pipeline([('extract_title', ExtractTitle(min_relative_freq = 0.2)),
                                   ('discretize_Fare', QCutFare(13)),
                                   ('fractional_Age', AddFractionalAgeColumn()),
                                   ('family_Size', AddSumOfColumnsColumn(columns=['SibSp','Parch'],
                                                                         constant=1,
                                                                         name='FamilySize')),
                                   ('drop', DropColumn(cols=['Ticket'])),
                                   ('prep',ColumnTransformer([('impute', 
                                                               SimpleImputer(strategy='mean'), 
                                                               ['Age']),
                                                              ('cabin_extract', 
                                                               extract_cabin, 
                                                               'Cabin'),
                                                              ('encode', 
                                                               OneHotEncoder(handle_unknown='ignore',
                                                                            sparse_output=False), 
                                                               ['Sex', 'Embarked', 'Title']),
                                                             ],
                                                             remainder='passthrough')),
                                   ])
preprocessing_pipeline

In [ ]:
pd.DataFrame(preprocessing_pipeline.fit_transform(train.drop(['Survived'], axis=1)))

In [ ]:
pd.DataFrame(preprocessing_pipeline.transform(test))

Great! the transformers produce a clean and usable data frame applied to both the training and test data. There is one small problem, though, and that is that the column titles have disappeared. Whereas the column titles are retained with the custom transformers that we defined ourselves:

In [ ]:
Pipeline(preprocessing_pipeline.steps[:-1]).fit_transform(train.drop(['Survived'], axis=1))

the `ColumnTransformer` and attached transformers (`SimpleImputer`, etc.) throw away the column names and return numpy arrays rather than pandas dataframes. This is OK when we pass the array through to a model, but is less than ideal for further data exploration, or indeed interpretation of the model outputs (such as feature importances, etc.). Furthermore, `ColumnTransformer` reorders the output so that we can't even easily match up the columns in the output data with the column names in the input data.

This has been a big problem for scikit-learn transformers and pipelines (see, for example, [here](https://datascience.stackexchange.com/q/75449/135593), [here](https://stackoverflow.com/q/54646709) and [here](https://stackoverflow.com/q/61079602)). Thankfully, however, this functionality has finally been implemented with the [set_output API](https://scikit-learn.org/stable/auto_examples/release_highlights/plot_release_highlights_1_2_0.html) in release 1.2 of scikit-learn.

We'll need to rewrite the `extract_cabin` transformer from section 6.2 using `FunctionTransformer` to return a data frame (ignore the warning):

In [ ]:
extract_cabin_named = FunctionTransformer(func = lambda df: df.apply(lambda col: [ord(x[0]) - ord('A') + 1 if type(x) == str else 0 for x in col])).set_output(transform="pandas")
extract_cabin_named.fit_transform(train[['Cabin']])

Now, we can get named output by appending `.set_output(transform='pandas')` onto the `ColumnTransformer`. (The `set_output` method looks like it recursively applies itself to subtransformations.)

In [ ]:
named_preprocessing_pipeline = Pipeline([('extract_title', ExtractTitle(min_relative_freq = 0.2)),
                                         ('discretize_Fare', QCutFare(13)),
                                         ('fractional_Age', AddFractionalAgeColumn()),
                                         ('family_Size', AddSumOfColumnsColumn(columns=['SibSp','Parch'],
                                                                               constant=1,
                                                                               name='FamilySize')),
                                         ('drop', DropColumn(cols=['Ticket'])),
                                         ('prep',ColumnTransformer([('impute', 
                                                                          SimpleImputer(strategy='mean'), 
                                                                          ['Age']),
                                                                         ('cabin_extract', 
                                                                          extract_cabin_named, 
                                                                          ['Cabin']),
                                                                         ('encode', 
                                                                          OneHotEncoder(handle_unknown='ignore',
                                                                                        sparse_output=False), 
                                                                          ['Sex', 'Embarked', 'Title']),
                                                                        ],
                                                                        remainder='passthrough').set_output(transform='pandas')),
                                         ])
named_preprocessing_pipeline

In [ ]:
named_preprocessing_pipeline.fit_transform(train.drop(['Survived'], axis=1))

# <p style="background-color:darkred;color:white;font-family:verdana;font-size:120%;text-align:center;border-radius: 15px 50px;">10. Modelling using the pipeline</p>

The final step is attaching a model onto the end of the pipeline. Let's use `LogisticRegression` again, with a normalization step just before the model:

In [ ]:
modelling_pipeline = Pipeline(named_preprocessing_pipeline.steps + \
                              [('scale',StandardScaler().set_output(transform='pandas')),
                               ('logreg', LogisticRegression(max_iter=2500))])
modelling_pipeline

In [ ]:
modelling_pipeline.fit(X=train.drop(['Survived'], axis=1),
                       y=train['Survived'])
predictions = modelling_pipeline.predict(test)
sample['Survived'] = predictions
sample.to_csv('accuracy_submission.csv', index=False)

In [ ]:
pipeline_cv = cross_validate(estimator = modelling_pipeline,
                             X = train.drop(['Survived'], axis=1),
                             y = train['Survived'],
                             cv = skf,
                             scoring = 'accuracy')
print(f"Average cross-validated accuracy from final pipeline: {pipeline_cv['test_score'].mean():.3f}")

Let's have a look at feature importances (defined for the logistic regression model as absolute value of the coefficients on scaled input data). Note that we only get column names from the output by using the `set_output` method as described in section 9.

In [ ]:
plt.figure(figsize=(6,10))
modelling_pipeline.fit(trainX, trainy)

x_plt = modelling_pipeline.steps[-1][1].coef_.ravel()
x_plt_abs = np.abs(x_plt)
y_plt = Pipeline(modelling_pipeline.steps[:-1]).fit_transform(trainX).columns
_,xp,yp=list(zip(*sorted(list(zip(x_plt_abs, x_plt, y_plt)), 
                                  reverse=True)))

_=sns.barplot(x=list(xp),y=list(yp))

We see that young, female passengers paying a larger fare are more likely to survive compared to poor, male passengers

<div class="alert alert-block alert-danger">
    <b>Spoiler Alert:</b> This is the plot of the movie Titanic</div>



Notice, however, that the 'FractionalAge' feature that we lovingly engineered doesn't look to be particularly useful. Maybe we should get rid of it. One way to drop this preprocessing step without re-writing the pipeline is to use the `set_params()` method of the pipeline:

> `modelling_pipeline.set_params(fractional_Age=None)`

But, since everything is in a pipeline, we can explicitly test the effect on the cross-validated score of keeeping or dropping this (or any other) preprocessing step.

# <p style="background-color:darkred;color:white;font-family:verdana;font-size:120%;text-align:center;border-radius: 15px 50px;">11. Testing preprocessing choices using GridSearchCV</p>

One of the great advantages of setting up preprocessing in a pipeline like this is that we can use `GridSearchCV` to tune the preprocessing choices in conjunction with any model hyperparameters. The [pipeline documentation](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) specifies how to do this:

> The purpose of the pipeline is to assemble several steps that can be cross-validated together while setting different parameters. For this, it enables setting parameters of the various steps using their names and the parameter name separated by a '__', as in the example below. A step’s estimator may be replaced entirely by setting the parameter with its name to another estimator, or a transformer removed by setting it to 'passthrough' or None.

For this example, I:

1. tune the regularization parameter of the logistic regression, with 'logreg__C' parameter (the 'logreg' is from the name associated with the `LogisticRegression` step of the pipeline, and the 'C' refers to the logistic regression regularization parameter ([see the LogisticRegression documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)),

2. choose whether the fractional age indicator feature engineering should be included (setting a step to `None` turns this step off), 

3. swap out the mean imputation strategy for a median imputation strategy (notice the chain of parameter naming to access the `SimpleImputer` strategy here), and

4. Choose the number of bins to discretize 'Fare' with.

We can get a list of the tunable parameters (and their names) by calling `get_params()` on the pipeline.

In [ ]:
%%time
from sklearn.model_selection import GridSearchCV
gscv_roc = GridSearchCV(estimator = modelling_pipeline,
                        cv = skf,
                        scoring='roc_auc',
                        verbose=0,
                        param_grid = {'logreg__C': [0.001,1,1000],
                                      'fractional_Age': [None,AddFractionalAgeColumn()],
                                      'prep__impute__strategy': ['mean', 'median'],
                                      'discretize_Fare__n_bins': [5,13],
                                     },
                       )
_=gscv_roc.fit(X = train.drop(['Survived'], axis=1),
               y = train['Survived'],
              )

Note that I changed the scoring metric here to [ROC_AUC](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html) just to differentiate between the scores a bit better, as accuracy tends to be very similar for the titanic data. We can see the effect of our different preprocessing from the mean test score variable from the `GridSearchCV.cv_results_` object:

In [ ]:
cv_res = pd.DataFrame({k:[[str(y)[:6] for y in x] for x in [gscv_roc.cv_results_['param_'+param_name].data for \
                                      param_name in gscv_roc.param_grid]][i] for \
                 i,k in enumerate(gscv_roc.param_grid)})
cv_res['mean_cross_validated_ROC_AUC'] = gscv_roc.cv_results_['mean_test_score']
cv_res['rank'] = gscv_roc.cv_results_['rank_test_score']
cv_res.to_csv('accuracy_cv.csv', index=False)
cv_res.sort_values(by = ['rank'])

In [ ]:
for pname in gscv_roc.param_grid:
    print(f'--- {pname} ---')
    print(cv_res.groupby(pname)['mean_cross_validated_ROC_AUC'].mean())
    print('')

So, we see that the fractional age indicator variable doesn't help predictions, mean imputation is slightly better than median imputation, and discretizing Fare into 5 bins rather than 13 improves our model.

# <p style="background-color:darkred;color:white;font-family:verdana;font-size:120%;text-align:center;border-radius: 15px 50px;">12. Limitations of pipelines</p>

Although pipelines have a lot of flexibility, there are a few limitations:

1. Preprocessing pipelines can end up being very CPU-intensive. If we are testing a lot of model parameters with exactly the same preprocessing steps, it will probably be worthwhile splittling the pipeline into two, and running the preprocessing pipeline only once. `Pipeline` has a 'memory' option, which can cache preprocessing steps, but I have not got around to using it properly.

2. The output from pipeline and transformers is an array rather than pandas dataframe. This means that:
    1. we can't chain processors by name in a pipeline. For example, the 'Cabin' feature extraction (section 6) returns an array rather than a named pandas `Series`. If we wanted to encode this in a separate, downstream, step (for example, along with the 'Sex' and 'Embarked' columns), we would have to refer to the column numbers in the `ColumnTransformer` step rather than the dataframe column names. `ColumnTransformer` also reorders columns (returning the transformed columns in order of the transformers and with the pass-through columns at the end), so it is difficult to tell which column number refers to which transformed column.
    2. we can't use column names from the output of a `ColumnTransformer` or `Pipeline` e.g. for feature importance. Model inference (e.g. feature importance scores, or even regression parameter standard errors) is hard to do when the column names are lost from the preprocessing.
    
3. Although pipelines (and transformers) accept the target variable `y`, which can in theory be used to transform the array of predictive variables `X`, [pipelines are unable to modify the target](https://stackoverflow.com/questions/25539311/custom-transformer-for-sklearn-pipeline-that-alters-both-x-and-y). Why would we want to do this? I recently wrote [a notebook on using EM for target imputation](https://www.kaggle.com/code/nnjjpp/imputing-censored-data-using-em). It would've been good to have this in a pipeline to test the effect of target imputation, but it appears to be currently impossible, without modifying the pipeline API: the [stackoverflow answer by Marco Cerliani](https://stackoverflow.com/a/70191787) provides a workaround by over-riding the sklearn `Pipeline` class, which works quite well, but it's a bit fiddly to use with custom transformers.


# <p style="background-color:darkred;color:white;font-family:verdana;font-size:120%;text-align:center;border-radius: 15px 50px;">13. Conclusions</p>

Pipelines are a way of organising data processing and/or modelling steps into one object. This is a better way of organising your data-science workflow. 

Almost all preprocessing steps can be written as a pipeline steps, and these include: scaling and transforming numerical data, imputation, encoding categorical data, replacing and adding columns (feature engineering), as well as dropping columns. 

Learning how to create your own custom scikit-learn transformers allows almost anything to be included in a pipeline.

There are several benefits of using a pipeline, and these include:

1. Eliminating duplicated code.

2. Avoiding data leakage by processing training and test data separately.

3. Embedding preprocessing in a cross-validation scheme.

4. Code readability.

5. Using pipelines allows the possibility of tuning preprocessing choices for better test predictions.




---

### Notebook version history

V10: Included `set_output` method for named pipeline output (section 9).

V11: Removed `StandardScalerNamed` transformer (we can use `set_output` on `StandardScaler` directly), and I added to the final discussion point.

V13: Included section on `make_column_selector` (section 5.1)

V14: typo